# **Bike Sharing Systems - Exploratory Data Analysis with SQL**

This notebook explores the Seoul bike sharing demand dataset and supporting
reference data using SQL queries via RSQLite. Key questions addressed:

- What are the basic characteristics of the demand dataset?
- How does demand vary by season, hour, and weather conditions?
- How does Seoul's bike sharing system compare to similarly scaled cities globally?

Results from this analysis feed directly into the modeling and dashboard steps
of the pipeline.

## **Setup**

We use RSQLite to run SQL queries directly in R without needing an external
database. All four processed datasets are loaded into an in-memory database
and queried from there.

In [ ]:
install.packages("RSQLite")

library(RSQLite)
library(tidyverse)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [ ]:
conn <- dbConnect(SQLite(), ":memory:")

# Load processed datasets into the database
seoul_bike_sharing      <- read_csv("seoul_bike_sharing.csv",      show_col_types = FALSE)
cities_weather_forecast <- read_csv("cities_weather_forecast.csv", show_col_types = FALSE)
bike_sharing_systems    <- read_csv("bike_sharing_systems.csv",    show_col_types = FALSE)
world_cities            <- read_csv("world_cities.csv",             show_col_types = FALSE)

dbWriteTable(conn, "SEOUL_BIKE_SHARING",      seoul_bike_sharing,      overwrite = TRUE)
dbWriteTable(conn, "CITIES_WEATHER_FORECAST", cities_weather_forecast, overwrite = TRUE)
dbWriteTable(conn, "BIKE_SHARING_SYSTEMS",    bike_sharing_systems,    overwrite = TRUE)
dbWriteTable(conn, "WORLD_CITIES",            world_cities,            overwrite = TRUE)

cat("Tables loaded:", paste(dbListTables(conn), collapse = ", "))

Tables loaded: BIKE_SHARING_SYSTEMS, CITIES_WEATHER_FORECAST, SEOUL_BIKE_SHARING, WORLD_CITIES

## **Record Count**

Confirm the number of records in the Seoul bike sharing dataset after
missing value removal in the wrangling step.

In [ ]:
dbGetQuery(conn, "
  SELECT COUNT(*) AS RECORD_COUNT
  FROM SEOUL_BIKE_SHARING
")

RECORD_COUNT
<int>
8465


## **Operational Hours**

Count the number of hours with non-zero bike rentals, i.e. hours where
the system was actively used.

In [ ]:
dbGetQuery(conn, "
  SELECT COUNT(*) AS OPERATIONAL_HOURS
  FROM SEOUL_BIKE_SHARING
  WHERE RENTED_BIKE_COUNT > 0
")

OPERATIONAL_HOURS
<int>
8465


## **Weather Outlook for Seoul**

Pull the next available 3-hour weather forecast for Seoul from the
cities weather forecast dataset.

In [ ]:
dbGetQuery(conn, "
  SELECT *
  FROM CITIES_WEATHER_FORECAST
  WHERE CITY = 'Seoul'
  LIMIT 1
")

CITY,WEATHER,VISIBILITY,TEMP,TEMP_MIN,TEMP_MAX,PRESSURE,HUMIDITY,WIND_SPEED,WIND_DEG,SEASON,FORECAST_DATETIME
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
Seoul,Clear,10000,12.32,10.91,12.32,1015,50,2.18,248,Spring,1618574400


## **Seasons**

Identify which seasons are represented in the Seoul bike sharing dataset.

In [ ]:
dbGetQuery(conn, "
  SELECT DISTINCT SEASONS
  FROM SEOUL_BIKE_SHARING
  ORDER BY SEASONS
")

SEASONS
<chr>
Autumn
Spring
Summer
Winter


## **Date Range**

Find the earliest and latest dates in the dataset to confirm the time
period covered.

In [ ]:
dbGetQuery(conn, "
  SELECT
    MIN(DATE(SUBSTR(DATE, 7, 4) || '-' || SUBSTR(DATE, 4, 2) || '-' || SUBSTR(DATE, 1, 2))) AS FIRST_DATE,
    MAX(DATE(SUBSTR(DATE, 7, 4) || '-' || SUBSTR(DATE, 4, 2) || '-' || SUBSTR(DATE, 1, 2))) AS LAST_DATE
  FROM SEOUL_BIKE_SHARING
")

FIRST_DATE,LAST_DATE
<chr>,<chr>
2017-12-01,2018-11-30


## **All-Time Peak Demand**

Identify the specific date and hour with the highest number of bike
rentals on record.

In [ ]:
dbGetQuery(conn, "
  SELECT DATE, HOUR, RENTED_BIKE_COUNT
  FROM SEOUL_BIKE_SHARING
  WHERE RENTED_BIKE_COUNT = (
    SELECT MAX(RENTED_BIKE_COUNT)
    FROM SEOUL_BIKE_SHARING
  )
")

DATE,HOUR,RENTED_BIKE_COUNT
<chr>,<dbl>,<dbl>
19/06/2018,18,3556


## **Hourly Demand and Temperature by Season**

Average hourly bike rentals and temperature grouped by season and hour,
ranked by average bike count. Shows which season and hour combinations
drive the highest demand.

In [ ]:
dbGetQuery(conn, "
  SELECT
    SEASONS,
    HOUR,
    ROUND(AVG(RENTED_BIKE_COUNT), 1) AS AVG_BIKE_COUNT,
    ROUND(AVG(TEMPERATURE), 1)       AS AVG_TEMPERATURE
  FROM SEOUL_BIKE_SHARING
  GROUP BY SEASONS, HOUR
  ORDER BY AVG_BIKE_COUNT DESC
  LIMIT 10
")

SEASONS,HOUR,AVG_BIKE_COUNT,AVG_TEMPERATURE
<chr>,<dbl>,<dbl>,<dbl>
Summer,18,2135.1,29.4
Autumn,18,1983.3,16.0
Summer,19,1889.3,28.3
Summer,20,1801.9,27.1
Summer,21,1754.1,26.3
Spring,18,1689.3,16.0
Summer,22,1567.9,25.7
Autumn,17,1562.9,17.3
Summer,17,1526.3,30.1


## **Rental Seasonality**

Summary statistics for bike rentals by season, including mean, min, max,
and standard deviation. Reveals how demand distribution shifts across seasons.

In [ ]:
dbGetQuery(conn, "
  SELECT
    SEASONS,
    ROUND(AVG(RENTED_BIKE_COUNT), 1)                                         AS AVG_BIKE_COUNT,
    MIN(RENTED_BIKE_COUNT)                                                    AS MIN_BIKE_COUNT,
    MAX(RENTED_BIKE_COUNT)                                                    AS MAX_BIKE_COUNT,
    ROUND(SQRT(AVG(RENTED_BIKE_COUNT * RENTED_BIKE_COUNT) -
          AVG(RENTED_BIKE_COUNT) * AVG(RENTED_BIKE_COUNT)), 1)               AS STD_BIKE_COUNT
  FROM SEOUL_BIKE_SHARING
  GROUP BY SEASONS
  ORDER BY AVG_BIKE_COUNT DESC
")

SEASONS,AVG_BIKE_COUNT,MIN_BIKE_COUNT,MAX_BIKE_COUNT,STD_BIKE_COUNT
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Summer,1034.1,9,3556,690.1
Autumn,924.1,2,3298,617.4
Spring,746.3,2,3251,618.5
Winter,225.5,3,937,150.3


## **Weather Seasonality**

Average weather conditions and bike demand per season, ranked by average
bike count. Provides an initial view of which weather factors correlate
with rental demand.

In [ ]:
dbGetQuery(conn, "
  SELECT
    SEASONS,
    ROUND(AVG(RENTED_BIKE_COUNT), 1)     AS AVG_BIKE_COUNT,
    ROUND(AVG(TEMPERATURE), 1)           AS AVG_TEMPERATURE,
    ROUND(AVG(HUMIDITY), 1)              AS AVG_HUMIDITY,
    ROUND(AVG(WIND_SPEED), 1)            AS AVG_WIND_SPEED,
    ROUND(AVG(VISIBILITY), 1)            AS AVG_VISIBILITY,
    ROUND(AVG(DEW_POINT_TEMPERATURE), 1) AS AVG_DEW_POINT,
    ROUND(AVG(SOLAR_RADIATION), 4)       AS AVG_SOLAR_RADIATION,
    ROUND(AVG(RAINFALL), 4)              AS AVG_RAINFALL,
    ROUND(AVG(SNOWFALL), 4)              AS AVG_SNOWFALL
  FROM SEOUL_BIKE_SHARING
  GROUP BY SEASONS
  ORDER BY AVG_BIKE_COUNT DESC
")

SEASONS,AVG_BIKE_COUNT,AVG_TEMPERATURE,AVG_HUMIDITY,AVG_WIND_SPEED,AVG_VISIBILITY,AVG_DEW_POINT,AVG_SOLAR_RADIATION,AVG_RAINFALL,AVG_SNOWFALL
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Summer,1034.1,26.6,65.0,1.6,1501.7,18.8,0.7613,0.2535,0.0000
Autumn,924.1,13.8,59.0,1.5,1558.2,5.2,0.5228,0.1177,0.0635
Spring,746.3,13.0,58.8,1.9,1240.9,4.1,0.6803,0.1869,0.0000
Winter,225.5,-2.5,49.7,1.9,1446.0,-12.4,0.2982,0.0328,0.2475


## **Seoul City Profile**

Join the world cities and bike sharing systems tables to retrieve Seoul's
geographic and demographic profile alongside its total bike fleet size.

In [ ]:
dbGetQuery(conn, "
  SELECT
    W.CITY,
    W.COUNTRY,
    W.LAT,
    W.LNG,
    W.POPULATION,
    SUM(B.BICYCLES) AS TOTAL_BICYCLES
  FROM WORLD_CITIES W, BIKE_SHARING_SYSTEMS B
  WHERE UPPER(W.CITY) = UPPER(B.CITY)
    AND UPPER(W.CITY) = 'SEOUL'
")

CITY,COUNTRY,LAT,LNG,POPULATION,TOTAL_BICYCLES
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Seoul,"Korea, South",37.5833,127,21794000,20000


## **Cities with Comparable Bike Fleet Size**

Find all cities with a total bike fleet between 15,000 and 20,000,
comparable in scale to Seoul's system. These cities will be used for
geographic visualization in the dashboard.

In [ ]:
dbGetQuery(conn, "
  SELECT
    W.CITY,
    W.COUNTRY,
    W.LAT,
    W.LNG,
    W.POPULATION,
    SUM(B.BICYCLES) AS TOTAL_BICYCLES
  FROM WORLD_CITIES W, BIKE_SHARING_SYSTEMS B
  WHERE UPPER(W.CITY) = UPPER(B.CITY)
  GROUP BY W.CITY, W.COUNTRY, W.LAT, W.LNG, W.POPULATION
  HAVING TOTAL_BICYCLES BETWEEN 15000 AND 20000
  ORDER BY TOTAL_BICYCLES DESC
")

CITY,COUNTRY,LAT,LNG,POPULATION,TOTAL_BICYCLES
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Seoul,"Korea, South",37.5833,127.0000,21794000,20000
Weifang,China,36.7167,119.1000,9373000,20000
Zhuzhou,China,27.8407,113.1469,3855609,20000
Shanghai,China,31.1667,121.4667,22120000,19165
Milan,Italy,45.4669,9.1900,1351562,17000
Milan,United States,35.9126,-88.7554,7201,17000
Milan,United States,42.0816,-83.6853,7778,17000
Beijing,China,39.9050,116.3914,19433000,16000
Ningbo,China,29.8750,121.5492,7639000,15000


In [ ]:
dbDisconnect(conn)